10. При снятии показаний измерительного прибора десятые доли деления шкалы прибора оцениваются «на глаз» наблюдателем. Количества цифр 0, 1, 2, …, 9, записанных наблюдателем в качестве десятых долей при 100 независимых измерениях, равны 5, 8, 6, 12, 14, 18, 11, 6, 13, 7 соответственно.

a) Проверить гипотезу о согласии данных с законом равномерного распределения с помощью критерия $\chi^2$ и с помощью критерия Колмогорова. Сравнить результаты.

b) Проверить гипотезу о согласии данных с законом нормального распределения с помощью критерия $\chi^2$ и с помощью критерия Колмогорова. Сравнить результаты.

In [1]:
#Подключаем нужные библиотеки
import numpy as np
import math
from scipy.special import kolmogorov
from scipy.stats import norm

In [2]:
mi = [5, 8, 6, 12, 14, 18, 11, 6, 13, 7]
N = 100


In [3]:
#a
F_emp = np.array([sum(mi[:i]) for i in range(len(mi) + 1)]) / N
F_theor = np.arange(10)/10

delta = np.sqrt(N) * np.max([max(abs(F_emp[i] - F_theor[i]), abs(F_emp[i+1] - F_theor[i])) for i in range(F_theor.size)])
p_value = kolmogorov(delta)
print(f"p-value по критерию Колмогорова для Н0:данные имеют равномерное распределение, составляет: {p_value}")
if p_value>0.05: print("=> Нет веских оснований отвергать гипотезу")
elif p_value < 0.05: print("=> Основная гипотеза отвергается")

p-value по критерию Колмогорова для Н0:данные имеют равномерное распределение, составляет: 0.039681879538114355
=> Основная гипотеза отвергается


In [4]:
#b
digits = np.arange(10)
data = np.sort(np.repeat(digits, mi))

mu = np.mean(data)
sigma = np.std(data)

intervals = [-np.inf, 0.5, 1.5, 2.5, 3.5, 4.5, 5.5, 6.5, 7.5, 8.5, np.inf]

p = []
for i in range(len(intervals) - 1):
    lower = intervals[i]
    upper = intervals[i+1]
    p_i = norm.cdf(upper, loc=mu, scale=sigma) - norm.cdf(lower, loc=mu, scale=sigma)
    p.append(p_i)

p = np.array(p)*100
print(f"Полученные npi: {p}")
mi_del = np.array(mi)
delta = np.sum(np.square(p - mi_del)/(p))
print(f"Δ = {delta}")


Полученные npi: [ 4.41616821  5.17552151  8.65410163 12.36537988 15.09787008 15.75250246
 14.04461048 10.7002627   6.96626253  6.82732051]
Δ = 10.798973140499792


In [5]:
bsp_iter = 50000

bootstrap_delta = []

delta_wave = np.sqrt(N) * np.max([max(np.abs(norm.cdf(digits[i], loc=mu, scale=sigma) - F_emp[i]),
                                      np.abs(norm.cdf(digits[i], loc=mu, scale=sigma) - F_emp[i + 1])) for i in digits])
F_bsp_emp_lower = [i / N for i in range(1, N + 1)]
F_bsp_emp_higher = [i / N for i in range(N)]
for _ in range(bsp_iter):
    random_sample = np.array(sorted(np.random.normal(mu, sigma, N)))
    
    a_bsp = random_sample.mean()
    sigma_bsp = random_sample.std()
    
    F_theor = norm.cdf(random_sample, loc=a_bsp, scale=sigma_bsp)
    d_1 = np.abs(F_theor - F_bsp_emp_lower)
    d_2 = np.abs(F_theor - F_bsp_emp_higher)
    sup = max(np.max(d_1), np.max(d_2))

    bootstrap_delta.append(np.sqrt(N) * sup)

bootstrap_delta = np.array(bootstrap_delta)

p_value = len(bootstrap_delta[bootstrap_delta >= delta_wave]) / bsp_iter

print(f"p-value по критерию Колмогорова для Н0:данные имеют нормальное распределение, составляет: {p_value}")
if p_value>0.05: print("=> Нет веских оснований отвергать гипотезу")
elif p_value < 0.05: print("=> Основная гипотеза отвергается")

p-value по критерию Колмогорова для Н0:данные имеют нормальное распределение, составляет: 0.01414
=> Основная гипотеза отвергается
